# 02 - Task 2: Variational Autoencoder (VAE) for Music Generation

**Author:** [Your Name]  
**Course:** CSE425/EEE474 — Unsupervised Neural Network for Multi-Genre Music Generation  
**Date:** Spring 2026

---

This notebook implements a Variational Autoencoder (VAE) that extends the basic autoencoder from Task 1.  
The VAE learns a structured latent space by enforcing a prior distribution, enabling smooth interpolation  
and meaningful music generation via random sampling.

**Key formulations:**
- Latent distribution: $q_\phi(z|X) = \mathcal{N}(\mu(X),\, \sigma(X))$
- Reparameterization trick: $z = \mu + \sigma \cdot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$
- Loss: $\mathcal{L}_{\text{VAE}} = \mathcal{L}_{\text{recon}} + \beta \cdot D_{\text{KL}}\big(q_\phi(z|X) \,\|\, p(z)\big)$

## Section 1 — Setup

Import all required libraries, set random seeds for reproducibility, define helper utilities,  
and load the preprocessed piano-roll data produced by `00_preprocessing.ipynb`.

### NVIDIA GPU (why training shows `cpu`)

A plain `pip install torch` from PyPI is usually **CPU-only** — then `torch.version.cuda` is `None` and `torch.cuda.is_available()` is `False`. Your RTX 4090 is fine; PyTorch simply was not installed with CUDA libraries.

**Fix:** run the **next code cell once** with `INSTALL_CUDA_PYTORCH = True`, then **Restart Kernel**, set it back to `False`, and **Run All** from the top.

RTX 40-series works well with **CUDA 12.x** wheels (`cu124`).

---

**If training crashes with `RuntimeError: CUDA error: unknown error`** (often during `MSELoss`): on Windows, **cuDNN is disabled by default** in the setup cell below so LSTM uses stable native GPU kernels. To try faster cuDNN again, set environment variable `PYTORCH_DISABLE_CUDNN=0` before launching Jupyter/Cursor.

In [ ]:
"""GPU bootstrap — run before training if `torch.version.cuda` is None."""

import shutil
import subprocess
import sys

# -----------------------------------------------------------------------------
# PyTorch from PyPI is CPU-only unless you install CUDA wheels from pytorch.org.
# Set True ONCE → run this cell → Restart Kernel → set False → Run All.
# -----------------------------------------------------------------------------
INSTALL_CUDA_PYTORCH = False


def _nvidia_smi_list():
    """Return human-readable GPU list from nvidia-smi, or an error string."""
    exe = shutil.which("nvidia-smi")
    if not exe:
        return False, "nvidia-smi not found on PATH — install NVIDIA drivers from https://www.nvidia.com/drivers/"
    try:
        proc = subprocess.run(
            [exe, "-L"],
            capture_output=True,
            text=True,
            timeout=20,
            check=False,
        )
        return True, proc.stdout.strip() or proc.stderr.strip()
    except Exception as exc:
        return False, str(exc)


def main():
    import torch

    cuda_compiled = torch.version.cuda  # None → CPU-only wheel
    cuda_ok = torch.cuda.is_available()

    print(f"torch.__version__ : {torch.__version__}")
    print(f"torch.version.cuda : {cuda_compiled}")
    print(f"torch.cuda.is_available() : {cuda_ok}")

    ok, gpu_txt = _nvidia_smi_list()
    print("\n--- nvidia-smi -L ---")
    print(gpu_txt)

    if cuda_compiled and cuda_ok:
        print("\nCUDA-enabled PyTorch is active — GPU training will work.")
        return

    if cuda_compiled and not cuda_ok:
        print(
            "\nPyTorch includes CUDA libraries but cuda.is_available() is False.\n"
            "Try updating drivers, reinstalling CUDA-enabled PyTorch, or checking env vars "
            "(CUDA_VISIBLE_DEVICES)."
        )
        return

    if not INSTALL_CUDA_PYTORCH:
        print(
            "\nCPU-only PyTorch detected.\n"
            "To use RTX 4090:\n"
            "  1) Set INSTALL_CUDA_PYTORCH = True in this cell.\n"
            "  2) Run this cell (downloads CUDA 12.4 PyTorch).\n"
            "  3) Restart Jupyter kernel.\n"
            "  4) Set INSTALL_CUDA_PYTORCH = False and Run All.\n"
            "\nManual alternative (PowerShell / terminal):\n"
            "  pip uninstall -y torch torchvision torchaudio\n"
            "  pip install torch torchvision torchaudio "
            "--index-url https://download.pytorch.org/whl/cu124"
        )
        return

    print("\nInstalling PyTorch + torchvision + torchaudio with CUDA 12.4 ...")
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--upgrade",
            "torch",
            "torchvision",
            "torchaudio",
            "--index-url",
            "https://download.pytorch.org/whl/cu124",
        ]
    )
    print(
        "\n>>> Done. Restart the Jupyter kernel now (Kernel → Restart).\n"
        ">>> Then set INSTALL_CUDA_PYTORCH = False and run all cells."
    )
    sys.exit(0)


main()


In [ ]:
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import pretty_midi
from sklearn.decomposition import PCA

# ---- Reproducibility & CUDA/cuDNN (aligned with notebook 01) --------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

_use_cuda = torch.cuda.is_available()
_env_cudnn = os.environ.get("PYTORCH_DISABLE_CUDNN")
if _env_cudnn is None:
    DISABLE_CUDNN_FOR_STABILITY = os.name == "nt"
else:
    DISABLE_CUDNN_FOR_STABILITY = _env_cudnn.strip().lower() not in ("0", "false", "no")

if _use_cuda:
    torch.cuda.manual_seed_all(SEED)
    if DISABLE_CUDNN_FOR_STABILITY:
        torch.backends.cudnn.enabled = False
    else:
        torch.backends.cudnn.benchmark = True
        torch.backends.cudnn.deterministic = False
else:
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 150, "savefig.dpi": 150})

device = torch.device("cuda" if _use_cuda else "cpu")

if _use_cuda:
    torch.cuda.empty_cache()
    w = torch.randn(256, 256, device=device, dtype=torch.float32)
    _warm_out = w @ w
    torch.cuda.synchronize()
    del w, _warm_out

print(f"Using device: {device}")
if _use_cuda:
    idx = torch.cuda.current_device()
    print(f"  GPU [{idx}]: {torch.cuda.get_device_name(idx)}")
    print(f"  CUDA (PyTorch built with): {torch.version.cuda}")
    mem_gb = torch.cuda.get_device_properties(idx).total_memory / (1024 ** 3)
    print(f"  VRAM available: {mem_gb:.1f} GB")
    print(f"  cuDNN enabled: {torch.backends.cudnn.enabled}")
else:
    print(f"  torch.version.cuda : {torch.version.cuda}")
    print("  Install CUDA-enabled PyTorch via the GPU bootstrap cell above.")


In [ ]:
def piano_roll_to_midi(piano_roll, fs=16, output_path=None):
    """Convert a (T, 128) piano-roll matrix to a PrettyMIDI object.

    Parameters
    ----------
    piano_roll : np.ndarray
        Binary or continuous piano-roll of shape (time_steps, 128).
    fs : int
        Sampling rate in frames-per-second used during preprocessing.
    output_path : str or None
        If provided, the MIDI file is written to this path.

    Returns
    -------
    pretty_midi.PrettyMIDI
        The resulting MIDI object.
    """
    midi = pretty_midi.PrettyMIDI()
    instrument = pretty_midi.Instrument(program=0)  # Acoustic Grand Piano

    binarized = (piano_roll > 0.5).astype(int)

    for pitch in range(128):
        active = False
        start_time = 0.0
        for t in range(binarized.shape[0]):
            if binarized[t, pitch] == 1 and not active:
                active = True
                start_time = t / fs
            elif binarized[t, pitch] == 0 and active:
                active = False
                end_time = t / fs
                note = pretty_midi.Note(
                    velocity=100, pitch=pitch,
                    start=start_time, end=end_time,
                )
                instrument.notes.append(note)
        if active:
            end_time = binarized.shape[0] / fs
            note = pretty_midi.Note(
                velocity=100, pitch=pitch,
                start=start_time, end=end_time,
            )
            instrument.notes.append(note)

    midi.instruments.append(instrument)

    if output_path is not None:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        midi.write(output_path)
        print(f"  -> Saved MIDI to {output_path}")

    return midi

### Load preprocessed data

In [ ]:
train_sequences = np.load('outputs/processed/train_sequences.npy')
val_sequences = np.load('outputs/processed/val_sequences.npy')
test_sequences = np.load('outputs/processed/test_sequences.npy')

print(f"Train sequences : {train_sequences.shape}")
print(f"Val   sequences : {val_sequences.shape}")
print(f"Test  sequences : {test_sequences.shape}")

### PyTorch Dataset & DataLoader

In [ ]:
class MusicDataset(Dataset):
    """Simple dataset wrapper for piano-roll numpy arrays."""

    def __init__(self, sequences: np.ndarray):
        """Initialise with an (N, seq_len, 128) numpy array."""
        self.data = torch.tensor(sequences, dtype=torch.float32)

    def __len__(self):
        """Return the number of sequences."""
        return len(self.data)

    def __getitem__(self, idx):
        """Return a single sequence tensor."""
        return self.data[idx]


BATCH_SIZE = 64

_pin_memory = torch.cuda.is_available()
_num_workers = min(4, max(1, (os.cpu_count() or 4) // 2))
if os.name == "nt":
    _num_workers = 0

train_dataset = MusicDataset(train_sequences)
val_dataset = MusicDataset(val_sequences)

_train_kw = dict(
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False,
    pin_memory=_pin_memory,
    num_workers=_num_workers,
)
_val_kw = dict(
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    pin_memory=_pin_memory,
    num_workers=_num_workers,
)
if _num_workers > 0:
    _train_kw["persistent_workers"] = True
    _train_kw["prefetch_factor"] = 2
    _val_kw["persistent_workers"] = True
    _val_kw["prefetch_factor"] = 2

train_loader = DataLoader(train_dataset, **_train_kw)
val_loader = DataLoader(val_dataset, **_val_kw)

print(f"Train batches: {len(train_loader)}")
print(f"Val   batches: {len(val_loader)}")
print(f"DataLoader: pin_memory={_pin_memory}, num_workers={_num_workers}")


## Section 2 — Model Architecture

The VAE consists of:

| Component | Details |
|---|---|
| **Encoder** | 2-layer bidirectional LSTM → 512-d hidden → two linear heads producing `mu` (64-d) and `logvar` (64-d) |
| **Decoder** | Linear expansion 64→256, repeated across 64 timesteps → 2-layer unidirectional LSTM → Linear 256→128 + Sigmoid |
| **Reparameterization** | $z = \mu + \exp(0.5 \cdot \log\sigma^2) \cdot \epsilon$ |

In [ ]:
class VAEEncoder(nn.Module):
    """Bidirectional LSTM encoder that maps a piano-roll sequence to (mu, logvar)."""

    def __init__(self, input_dim=128, hidden_dim=256, latent_dim=64, num_layers=2):
        """Initialise encoder layers.

        Parameters
        ----------
        input_dim : int
            Feature dimension per timestep (128 MIDI pitches).
        hidden_dim : int
            LSTM hidden size per direction.
        latent_dim : int
            Dimensionality of the latent space.
        num_layers : int
            Number of stacked LSTM layers.
        """
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
        )
        self.mu_layer = nn.Linear(hidden_dim * 2, latent_dim)
        self.logvar_layer = nn.Linear(hidden_dim * 2, latent_dim)

    def forward(self, x):
        """Encode input sequence and return (mu, logvar).

        Parameters
        ----------
        x : torch.Tensor
            Shape (batch, seq_len, 128).

        Returns
        -------
        mu : torch.Tensor  — shape (batch, latent_dim)
        logvar : torch.Tensor — shape (batch, latent_dim)
        """
        _, (h_n, _) = self.lstm(x)
        # h_n shape: (num_layers * 2, batch, hidden_dim)
        # Take the last layer's forward and backward hidden states
        h_forward = h_n[-2]  # (batch, hidden_dim)
        h_backward = h_n[-1]  # (batch, hidden_dim)
        h_cat = torch.cat([h_forward, h_backward], dim=1)  # (batch, hidden_dim*2)

        mu = self.mu_layer(h_cat)
        logvar = self.logvar_layer(h_cat)
        return mu, logvar

In [ ]:
class VAEDecoder(nn.Module):
    """LSTM decoder that reconstructs a piano-roll sequence from a latent vector z."""

    def __init__(self, latent_dim=64, hidden_dim=256, output_dim=128, seq_len=64, num_layers=2):
        """Initialise decoder layers.

        Parameters
        ----------
        latent_dim : int
            Dimensionality of the latent vector.
        hidden_dim : int
            LSTM hidden size.
        output_dim : int
            Feature dimension per timestep (128 MIDI pitches).
        seq_len : int
            Number of timesteps to reconstruct.
        num_layers : int
            Number of stacked LSTM layers.
        """
        super().__init__()
        self.seq_len = seq_len
        self.expand = nn.Linear(latent_dim, hidden_dim)
        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False,
        )
        self.output_layer = nn.Linear(hidden_dim, output_dim)

    def forward(self, z):
        """Decode latent vector into a reconstructed piano-roll.

        Parameters
        ----------
        z : torch.Tensor
            Shape (batch, latent_dim).

        Returns
        -------
        x_hat : torch.Tensor — shape (batch, seq_len, 128)
        """
        h = self.expand(z)  # (batch, hidden_dim)
        h = h.unsqueeze(1).repeat(1, self.seq_len, 1)  # (batch, seq_len, hidden_dim)
        lstm_out, _ = self.lstm(h)  # (batch, seq_len, hidden_dim)
        x_hat = torch.sigmoid(self.output_layer(lstm_out))  # (batch, seq_len, 128)
        return x_hat

In [ ]:
class MusicVAE(nn.Module):
    """Full VAE combining encoder, reparameterization, and decoder."""

    def __init__(self, input_dim=128, hidden_dim=256, latent_dim=64, seq_len=64, num_layers=2):
        """Initialise the MusicVAE.

        Parameters
        ----------
        input_dim : int
            Feature dimension per timestep.
        hidden_dim : int
            LSTM hidden size.
        latent_dim : int
            Latent space dimensionality.
        seq_len : int
            Sequence length.
        num_layers : int
            Stacked LSTM layers.
        """
        super().__init__()
        self.encoder = VAEEncoder(input_dim, hidden_dim, latent_dim, num_layers)
        self.decoder = VAEDecoder(latent_dim, hidden_dim, input_dim, seq_len, num_layers)

    def reparameterize(self, mu, logvar):
        """Sample z via the reparameterization trick.

        z = mu + exp(0.5 * logvar) * epsilon,  epsilon ~ N(0, I)
        """
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(mu)
        return mu + std * eps

    def forward(self, x):
        """Full forward pass: encode -> reparameterize -> decode.

        Returns
        -------
        x_hat, mu, logvar
        """
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decoder(z)
        return x_hat, mu, logvar

    def encode(self, x):
        """Encode input and return (mu, logvar) without sampling."""
        return self.encoder(x)

    def decode(self, z):
        """Decode a latent vector z into a reconstructed piano-roll."""
        return self.decoder(z)


model = MusicVAE().to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print(model)

## Section 3 — Loss Function

The VAE loss is the sum of:
1. **Reconstruction loss** — MSE between input and output piano-rolls.  
2. **KL divergence** — regularises the latent distribution toward $\mathcal{N}(0, I)$.  

A scalar $\beta$ controls the KL weight (beta-annealing is applied during training).

In [ ]:
def vae_loss(x, x_hat, mu, logvar, beta=1.0):
    """Compute the VAE loss: reconstruction + beta * KL divergence.

    Parameters
    ----------
    x : torch.Tensor
        Original input, shape (batch, seq_len, 128).
    x_hat : torch.Tensor
        Reconstruction, same shape as x.
    mu : torch.Tensor
        Latent mean, shape (batch, latent_dim).
    logvar : torch.Tensor
        Log-variance, shape (batch, latent_dim).
    beta : float
        Weight for the KL term.

    Returns
    -------
    total_loss, recon_loss, kl_loss : tuple of torch.Tensor (scalars)
    """
    recon_loss = F.mse_loss(x_hat, x, reduction='mean')
    kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    total_loss = recon_loss + beta * kl_loss
    return total_loss, recon_loss, kl_loss

## Section 4 — Training

Training uses **beta-annealing**: $\beta$ starts at 0 and linearly increases to 1.0 over the  
first 20 epochs. This lets the model first learn a good reconstruction and then gradually  
shape the latent space.

The best model (lowest validation total loss) is saved to `outputs/task2/best_model.pth`.
**Resume:** full state after each epoch → `outputs/task2/training_checkpoint.pth`. Set `RESUME_IF_AVAILABLE = True` in the training cell (default). Only completed epochs are saved.


In [ ]:
os.makedirs('outputs/task2', exist_ok=True)
os.makedirs('outputs/plots', exist_ok=True)

NUM_EPOCHS = 60
LEARNING_RATE = 1e-3
BETA_ANNEAL_EPOCHS = 20

RESUME_IF_AVAILABLE = True
CHECKPOINT_PATH = os.path.join("outputs", "task2", "training_checkpoint.pth")

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

history = {
    'epoch': [],
    'beta': [],
    'train_recon_loss': [],
    'train_kl_loss': [],
    'train_total_loss': [],
    'val_recon_loss': [],
    'val_kl_loss': [],
    'val_total_loss': [],
}

best_val_loss = float('inf')
start_epoch = 1

if RESUME_IF_AVAILABLE and os.path.isfile(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    history = {k: list(v) for k, v in ckpt["history"].items()}
    best_val_loss = float(ckpt["best_val_loss"])
    completed = int(ckpt["epoch"])
    start_epoch = completed + 1
    print(
        f"Resumed from checkpoint: epochs 1–{completed} done; "
        f"continuing from epoch {start_epoch}/{NUM_EPOCHS}"
    )
else:
    print("Starting VAE training from epoch 1 (no checkpoint or RESUME_IF_AVAILABLE=False).")

if start_epoch > NUM_EPOCHS:
    print(
        f"Training already finished for NUM_EPOCHS={NUM_EPOCHS}. "
        f"Raise NUM_EPOCHS or delete {CHECKPOINT_PATH}."
    )
else:
    for epoch in range(start_epoch, NUM_EPOCHS + 1):
        beta = min(1.0, epoch / BETA_ANNEAL_EPOCHS)

        # ---- Training -----------------------------------------------------------
        model.train()
        train_recon, train_kl, train_total = 0.0, 0.0, 0.0
        train_batches = 0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch:02d}/{NUM_EPOCHS} [train]", leave=False):
            batch = batch.to(device)
            x_hat, mu, logvar = model(batch)
            loss, r_loss, kl = vae_loss(batch, x_hat, mu, logvar, beta=beta)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_recon += r_loss.item()
            train_kl += kl.item()
            train_total += loss.item()
            train_batches += 1

        train_recon /= train_batches
        train_kl /= train_batches
        train_total /= train_batches

        # ---- Validation ---------------------------------------------------------
        model.eval()
        val_recon, val_kl, val_total = 0.0, 0.0, 0.0
        val_batches = 0

        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch:02d}/{NUM_EPOCHS} [val]  ", leave=False):
                batch = batch.to(device)
                x_hat, mu, logvar = model(batch)
                loss, r_loss, kl = vae_loss(batch, x_hat, mu, logvar, beta=beta)

                val_recon += r_loss.item()
                val_kl += kl.item()
                val_total += loss.item()
                val_batches += 1

        val_recon /= val_batches
        val_kl /= val_batches
        val_total /= val_batches

        # ---- Bookkeeping --------------------------------------------------------
        history['epoch'].append(epoch)
        history['beta'].append(beta)
        history['train_recon_loss'].append(train_recon)
        history['train_kl_loss'].append(train_kl)
        history['train_total_loss'].append(train_total)
        history['val_recon_loss'].append(val_recon)
        history['val_kl_loss'].append(val_kl)
        history['val_total_loss'].append(val_total)

        save_flag = ''
        if val_total < best_val_loss:
            best_val_loss = val_total
            torch.save(model.state_dict(), 'outputs/task2/best_model.pth')
            save_flag = ' ** saved **'

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "history": history,
                "best_val_loss": best_val_loss,
                "num_epochs": NUM_EPOCHS,
                "learning_rate": LEARNING_RATE,
                "beta_anneal_epochs": BETA_ANNEAL_EPOCHS,
            },
            CHECKPOINT_PATH,
        )

        print(
            f"Epoch {epoch:02d} | beta={beta:.3f} | "
            f"Train [recon={train_recon:.5f}  kl={train_kl:.5f}  total={train_total:.5f}] | "
            f"Val   [recon={val_recon:.5f}  kl={val_kl:.5f}  total={val_total:.5f}]{save_flag}"
        )

    print(f"\nTraining complete. Best val loss: {best_val_loss:.5f}")


In [ ]:
# Load the best model weights for all downstream evaluation
model.load_state_dict(torch.load('outputs/task2/best_model.pth', map_location=device))
model.eval()
print("Loaded best model weights.")

## Section 5 — Visualizations

We produce five diagnostic plots that are saved as high-resolution PNGs.

### Plot 1 — Training Loss Curves

Reconstruction loss, KL divergence, and total loss over training epochs.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
epochs = history['epoch']
ax.plot(epochs, history['train_recon_loss'], label='Reconstruction Loss', linewidth=2)
ax.plot(epochs, history['train_kl_loss'], label='KL Divergence', linewidth=2)
ax.plot(epochs, history['train_total_loss'], label='Total Loss', linewidth=2, linestyle='--')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Task 2 — VAE Training Loss Curves')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('outputs/plots/task2_loss_curves.png', dpi=150)
plt.show()
print("Saved -> outputs/plots/task2_loss_curves.png")

### Plot 2 — Beta Annealing Schedule

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history['epoch'], history['beta'], linewidth=2, color='tab:orange')
ax.set_xlabel('Epoch')
ax.set_ylabel('Beta (β)')
ax.set_title('Task 2 — Beta Annealing Schedule')
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='β = 1.0')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('outputs/plots/task2_beta_annealing.png', dpi=150)
plt.show()
print("Saved -> outputs/plots/task2_beta_annealing.png")

### Plot 3 — Latent Space Visualization (PCA)

Encode 500 validation samples and project their latent means into 2-D with PCA.

In [ ]:
num_vis_samples = min(500, len(val_dataset))
vis_loader = DataLoader(val_dataset, batch_size=num_vis_samples, shuffle=False)
vis_batch = next(iter(vis_loader)).to(device)

with torch.no_grad():
    mu_vis, _ = model.encode(vis_batch)
    mu_np = mu_vis.cpu().numpy()

pca = PCA(n_components=2)
z_2d = pca.fit_transform(mu_np)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(z_2d[:, 0], z_2d[:, 1], alpha=0.5, s=15, c=np.arange(len(z_2d)), cmap='viridis')
ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.set_title('Task 2 — Latent Space (PCA of μ, 500 val samples)')
cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label('Sample Index')
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('outputs/plots/task2_latent_pca.png', dpi=150)
plt.show()
print("Saved -> outputs/plots/task2_latent_pca.png")

### Plot 4 — Latent Space Interpolation

Take two random validation samples, encode them, linearly interpolate in 8 steps,  
and visualise the decoded piano-rolls as a grid.

In [ ]:
idx1, idx2 = np.random.choice(len(val_dataset), size=2, replace=False)
sample1 = val_dataset[idx1].unsqueeze(0).to(device)
sample2 = val_dataset[idx2].unsqueeze(0).to(device)

with torch.no_grad():
    mu1, _ = model.encode(sample1)
    mu2, _ = model.encode(sample2)

    alphas = np.linspace(0, 1, 8)
    interpolated_rolls = []
    for alpha in alphas:
        z_interp = mu1 + alpha * (mu2 - mu1)
        x_dec = model.decode(z_interp)
        interpolated_rolls.append(x_dec.squeeze(0).cpu().numpy())

fig, axes = plt.subplots(2, 4, figsize=(20, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(interpolated_rolls[i].T, aspect='auto', origin='lower', cmap='magma')
    ax.set_title(f'α = {alphas[i]:.2f}')
    ax.set_xlabel('Time Step')
    ax.set_ylabel('MIDI Pitch')
fig.suptitle('Task 2 — Latent Interpolation Between Two Samples', fontsize=14)
fig.tight_layout()
fig.savefig('outputs/plots/task2_interpolation.png', dpi=150)
plt.show()
print("Saved -> outputs/plots/task2_interpolation.png")

## Section 6 — Music Generation

Generate 8 music pieces by sampling $z \sim \mathcal{N}(0, I)$ and decoding.

In [ ]:
os.makedirs('outputs/task2/generated_midis', exist_ok=True)

num_generate = 8
generated_rolls = []

print("Generating MIDI files from random latent vectors ...")
for i in range(1, num_generate + 1):
    z_sample = torch.randn(1, 64).to(device)
    with torch.no_grad():
        x_gen = model.decode(z_sample).squeeze(0).cpu().numpy()
    generated_rolls.append(x_gen)

    midi_path = f'outputs/task2/generated_midis/generated_{i}.mid'
    piano_roll_to_midi(x_gen, fs=16, output_path=midi_path)

print(f"\nDone — {num_generate} MIDI files saved to outputs/task2/generated_midis/")

## Section 7 — Metric Comparison vs Task 1

For each generated piece we compute:

| Metric | Formula |
|---|---|
| **Rhythm Diversity** | $D_{\text{rhythm}} = \frac{\#\text{unique durations}}{\#\text{total notes}}$ |
| **Repetition Ratio** | $R = \frac{\#\text{repeated patterns}}{\#\text{total patterns}}$ (sliding window = 4 bars) |
| **Pitch Entropy** | Shannon entropy of a 12-bin pitch-class histogram |

We then compare these metrics against Task 1 results.

In [ ]:
def compute_rhythm_diversity(piano_roll, fs=16):
    """Compute rhythm diversity as the ratio of unique note durations to total notes.

    Parameters
    ----------
    piano_roll : np.ndarray
        Shape (T, 128), values binarised at 0.5 threshold.
    fs : int
        Frames per second.

    Returns
    -------
    float
        Rhythm diversity score in [0, 1].
    """
    binarized = (piano_roll > 0.5).astype(int)
    durations = []
    for pitch in range(128):
        active = False
        dur = 0
        for t in range(binarized.shape[0]):
            if binarized[t, pitch] == 1:
                if not active:
                    active = True
                    dur = 1
                else:
                    dur += 1
            else:
                if active:
                    durations.append(dur / fs)
                    active = False
        if active:
            durations.append(dur / fs)

    if len(durations) == 0:
        return 0.0
    return len(set(durations)) / len(durations)


def compute_repetition_ratio(piano_roll, bar_length=16, window=4):
    """Compute repetition ratio using a sliding window of `window` bars.

    Parameters
    ----------
    piano_roll : np.ndarray
        Shape (T, 128).
    bar_length : int
        Number of frames per bar.
    window : int
        Number of bars per pattern.

    Returns
    -------
    float
        Ratio of repeated patterns to total patterns.
    """
    binarized = (piano_roll > 0.5).astype(int)
    pattern_len = bar_length * window
    patterns = []
    for start in range(0, binarized.shape[0] - pattern_len + 1, bar_length):
        pattern = binarized[start:start + pattern_len].tobytes()
        patterns.append(pattern)

    if len(patterns) == 0:
        return 0.0

    seen = set()
    repeated = 0
    for p in patterns:
        if p in seen:
            repeated += 1
        seen.add(p)

    return repeated / len(patterns)


def compute_pitch_entropy(piano_roll):
    """Compute Shannon entropy of the 12-bin pitch-class histogram.

    Parameters
    ----------
    piano_roll : np.ndarray
        Shape (T, 128).

    Returns
    -------
    float
        Pitch entropy in bits.
    """
    binarized = (piano_roll > 0.5).astype(int)
    pitch_activity = binarized.sum(axis=0)  # (128,)

    chroma = np.zeros(12)
    for pitch in range(128):
        chroma[pitch % 12] += pitch_activity[pitch]

    total = chroma.sum()
    if total == 0:
        return 0.0

    probs = chroma / total
    probs = probs[probs > 0]
    entropy = -np.sum(probs * np.log2(probs))
    return entropy

In [ ]:
rows = []
for i, roll in enumerate(generated_rolls, start=1):
    rd = compute_rhythm_diversity(roll)
    rr = compute_repetition_ratio(roll)
    pe = compute_pitch_entropy(roll)
    rows.append({
        'sample': f'generated_{i}',
        'rhythm_diversity': rd,
        'repetition_ratio': rr,
        'pitch_entropy': pe,
    })
    print(f"Sample {i}: Rhythm Diversity={rd:.4f}  Repetition Ratio={rr:.4f}  Pitch Entropy={pe:.4f}")

task2_metrics = pd.DataFrame(rows)
task2_metrics.to_csv('outputs/task2/metrics.csv', index=False)
print(f"\nTask 2 metrics saved to outputs/task2/metrics.csv")
task2_metrics

### Comparison with Task 1

In [ ]:
task1_metrics = pd.read_csv('outputs/task1/metrics.csv')
print("Task 1 metrics:")
print(task1_metrics)
print()

# Compute means
t1_mean_rd = task1_metrics['rhythm_diversity'].mean()
t1_mean_rr = task1_metrics['repetition_ratio'].mean()
t1_mean_pe = task1_metrics['pitch_entropy'].mean()

t2_mean_rd = task2_metrics['rhythm_diversity'].mean()
t2_mean_rr = task2_metrics['repetition_ratio'].mean()
t2_mean_pe = task2_metrics['pitch_entropy'].mean()

comparison_df = pd.DataFrame({
    'Metric': ['Rhythm Diversity', 'Repetition Ratio', 'Pitch Entropy'],
    'Task 1 (AE)': [t1_mean_rd, t1_mean_rr, t1_mean_pe],
    'Task 2 (VAE)': [t2_mean_rd, t2_mean_rr, t2_mean_pe],
})

print("=" * 60)
print("       COMPARISON: Task 1 (AE) vs Task 2 (VAE)")
print("=" * 60)
print(comparison_df.to_string(index=False))
print("=" * 60)

### Plot 5 — Side-by-Side Bar Chart (Task 1 vs Task 2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

metrics_labels = ['Rhythm Diversity', 'Repetition Ratio', 'Pitch Entropy']
task1_vals = [t1_mean_rd, t1_mean_rr, t1_mean_pe]
task2_vals = [t2_mean_rd, t2_mean_rr, t2_mean_pe]

x = np.arange(len(metrics_labels))
bar_width = 0.3

bars1 = ax.bar(x - bar_width / 2, task1_vals, bar_width, label='Task 1 — AE', color='steelblue')
bars2 = ax.bar(x + bar_width / 2, task2_vals, bar_width, label='Task 2 — VAE', color='coral')

ax.set_xlabel('Metric')
ax.set_ylabel('Score')
ax.set_title('Task 1 (Autoencoder) vs Task 2 (VAE) — Mean Metric Comparison')
ax.set_xticks(x)
ax.set_xticklabels(metrics_labels)
ax.legend()
ax.grid(axis='y', alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

fig.tight_layout()
fig.savefig('outputs/plots/task1_vs_task2_comparison.png', dpi=150)
plt.show()
print("Saved -> outputs/plots/task1_vs_task2_comparison.png")

---

**End of Notebook — Task 2 (VAE) complete.**  
Proceed to Task 3 for the GAN-based approach.